# Data Preparation
Load raw data, compute lead_time, map factories, geocode customers, compute distances, handle outliers, save to data/processed/.


In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from src.config import *
import pandas as pd

%load_ext autoreload
%autoreload 2
from src.features import *

df = pd.read_csv(RAW_FILE)
print(df.shape)
print(df.columns.tolist())
print(df.head(3).to_string())

(10194, 18)
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Country/Region', 'City', 'State/Province', 'Postal Code', 'Division', 'Region', 'Product ID', 'Product Name', 'Sales', 'Units', 'Gross Profit', 'Cost']
   Row ID                      Order ID  Order Date   Ship Date       Ship Mode  Customer ID Country/Region        City State/Province Postal Code   Division    Region     Product ID                       Product Name  Sales  Units  Gross Profit  Cost
0       1  US-2021-103800-CHO-MIL-31000  03-01-2024  30-06-2026  Standard Class       103800  United States     Houston          Texas       77095  Chocolate  Interior  CHO-MIL-31000         Wonka Bar - Milk Chocolate   6.50      2          4.22  2.28
1       2  US-2021-112326-CHO-TRI-54000  04-01-2024  01-07-2026  Standard Class       112326  United States  Naperville       Illinois       60540  Chocolate  Interior  CHO-TRI-54000  Wonka Bar - Triple Dazzle Caramel   7.50      2          4.90  2.60
2

In [2]:
# Cell 2: inspect raw data before cleaning
raw = load_raw()
print(raw.dtypes)
print(raw.isna().sum())
print("Duplicates:", raw.duplicated().sum())
print(raw[COL_SHIP_MODE].value_counts())
print(raw[COL_REGION].value_counts())
print(sorted(raw[COL_STATE].dropna().unique()))

Row ID              int64
Order ID              str
Order Date            str
Ship Date             str
Ship Mode             str
Customer ID         int64
Country/Region        str
City                  str
State/Province        str
Postal Code           str
Division              str
Region                str
Product ID            str
Product Name          str
Sales             float64
Units               int64
Gross Profit      float64
Cost              float64
dtype: object
Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Country/Region    0
City              0
State/Province    0
Postal Code       0
Division          0
Region            0
Product ID        0
Product Name      0
Sales             0
Units             0
Gross Profit      0
Cost              0
dtype: int64
Duplicates: 0
Ship Mode
Standard Class    6120
Second Class      1979
First Class       1548
Same Day           547
Name: count, dtype: int64
Reg

In [3]:
# Cell 3: build the clean dataset
df = build_clean()
validate(df)
df.head()

d:\nassau-candy-optimizer\src\features.py:38: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[COL_SHIP_DATE] = pd.to_datetime(df[COL_SHIP_DATE], dayfirst=DATE_DAYFIRST, errors="coerce")


Rows with unparseable dates: 6064
Lead-time cap (99th pct): 1932 days
Rows: 10194 -> 4090
All validation checks passed.


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,...,ship_mode,region,division,units,sales,gross_profit,product,state,order_ordinal,order_month
0,1,US-2021-103800-CHO-MIL-31000,2024-03-01,2026-06-30,Standard Class,103800,United States,Houston,Texas,77095,...,Standard Class,Interior,Chocolate,2,6.50,4.22,Wonka Bar - Milk Chocolate,Texas,1521,3
1,2,US-2021-112326-CHO-TRI-54000,2024-04-01,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,Standard Class,Interior,Chocolate,2,7.50,4.90,Wonka Bar - Triple Dazzle Caramel,Illinois,1552,4
2,3,US-2021-112326-CHO-NUT-13000,2024-04-01,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,Standard Class,Interior,Chocolate,3,10.47,7.47,Wonka Bar - Nutty Crunch Surprise,Illinois,1552,4
3,4,US-2021-112326-CHO-SCR-58000,2024-04-01,2026-07-01,Standard Class,112326,United States,Naperville,Illinois,60540,...,Standard Class,Interior,Chocolate,3,10.80,7.50,Wonka Bar -Scrumdiddlyumptious,Illinois,1552,4
4,5,US-2021-141817-CHO-TRI-54000,2024-05-01,2026-07-05,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,...,Standard Class,Atlantic,Chocolate,3,11.25,7.35,Wonka Bar - Triple Dazzle Caramel,Pennsylvania,1582,5


In [4]:
# Cell 4: sanity checks
print(df[TARGET].describe())
print(df.groupby("factory")["distance_km"].describe()[["count", "mean", "min", "max"]])
print(df["factory"].value_counts())

count    4090.000000
mean     1360.218337
std       298.560261
min       613.000000
25%      1155.000000
50%      1361.000000
75%      1592.000000
max      1932.000000
Name: lead_time, dtype: float64
                    count         mean          min          max
factory                                                         
Lot's O' Nuts      2343.0  2127.778716    99.338479  4796.586172
Secret Factory       93.0  1437.458102   180.263094  2579.338892
Sugar Shack           9.0  1658.327064  1035.787446  2059.840188
The Other Factory    32.0  1525.925218   305.200675  2939.692735
Wicked Choccy's    1613.0  1861.103831   198.457966  3796.261990
factory
Lot's O' Nuts        2343
Wicked Choccy's      1613
Secret Factory         93
The Other Factory      32
Sugar Shack             9
Name: count, dtype: int64
